# React 框架 · 第 07 天

## 极客头条：登录鉴权、Layout 与首页图表

**资料范围**：以 day07 PPT 的讲授顺序为主，结合配套总笔记中的“登录模块”“Layout 模块”和 `react-jike` 代码。day06～day09 虽共用同一套总笔记，day07 课件重点是 Token 注入、路由鉴权、后台布局、用户信息、退出登录、401 处理和 ECharts。本篇按这些内容整理。

## 一、本日学习路线

```text
登录后已有 token
  → Axios 请求注入 token
  → AuthRoute 保护后台路由
  → Layout + 二级路由组织页面
  → 菜单跳转与反向高亮
  → 个人信息读取 / 退出登录 / 401 失效处理
  → 首页 ECharts 与接口模块封装
```

完成后应能解释“页面进入前鉴权”和“请求后 401 处理”的区别，并让菜单、路由、用户状态和图表形成完整后台骨架。

## 二、请求拦截器统一注入 Token

后台的受保护接口通常要求 `Authorization: Bearer <token>`。Token 已在前一天登录流程中持久化，可由 Axios 请求拦截器统一读取：

```js
import axios from 'axios'
import { getToken } from './token'

const request = axios.create({
  baseURL: 'http://geek.itheima.net/v1_0',
  timeout: 5000,
})

request.interceptors.request.use(config => {
  const token = getToken()
  if (token) config.headers.Authorization = `Bearer ${token}`
  return config
})
```

这样业务模块只需调用 `request.get(...)`，不必每个请求都手写请求头。检查 Network 的 Request Headers，可确认 Token 是否真正随请求发送。**不要在示例日志或截图中公开真实 Token。**

![Token 请求流程](assets/day07/07.png)

## 三、路由进入前的鉴权：`AuthRoute`

请求头有 Token 不代表页面已受到保护。未经登录的用户若直接输入后台地址，应在渲染 Layout 前被重定向：

```jsx
import { Navigate } from 'react-router-dom'
import { getToken } from '@/utils/token'

function AuthRoute({ children }) {
  return getToken()
    ? children
    : <Navigate to="/login" replace />
}
```

```jsx
{
  path: '/',
  element: (
    <AuthRoute>
      <Layout />
    </AuthRoute>
  ),
  children: [/* 子路由 */],
}
```

`replace` 避免浏览器返回键回到被拦截的页面。这个检查只能提高前端体验；真正的访问权限仍由服务端验证 Token。

## 四、搭建后台 Layout

Ant Design 的 `Layout`、`Header`、`Sider`、`Menu` 可以拼出“顶部信息区 + 左侧菜单 + 右侧内容区”：

```jsx
import { Layout, Menu } from 'antd'
import { Outlet } from 'react-router-dom'

const { Header, Sider, Content } = Layout

function GeekLayout() {
  return (
    <Layout className="geek-layout">
      <Header className="header">头条后台</Header>
      <Layout>
        <Sider width={200}>
          <Menu mode="inline" theme="dark" items={items} />
        </Sider>
        <Content className="layout-content">
          <Outlet />
        </Content>
      </Layout>
    </Layout>
  )
}
```

`Outlet` 是子路由渲染出口。若路由表配置了 children，但 Layout 没有 Outlet，就会只显示菜单而看不到 Home、Article 等页面。页面占满视口还需将 `html`、`body`、`#root` 设为 `height: 100%`。

## 五、二级路由与侧边菜单

Layout 下配置首页、文章管理、创建文章三个子页面：

```jsx
const router = createBrowserRouter([
  {
    path: '/',
    element: <AuthRoute><GeekLayout /></AuthRoute>,
    children: [
      { index: true, element: <Home /> },
      { path: 'article', element: <Article /> },
      { path: 'publish', element: <Publish /> },
    ],
  },
  { path: '/login', element: <Login /> },
])
```

菜单项的 `key` 直接使用路由地址，点击后导航：

```jsx
const items = [
  { key: '/', label: '首页', icon: <HomeOutlined /> },
  { key: '/article', label: '文章管理', icon: <DiffOutlined /> },
  { key: '/publish', label: '创建文章', icon: <EditOutlined /> },
]

const navigate = useNavigate()
<Menu items={items} onClick={({ key }) => navigate(key)} />
```

菜单路径和路由表要一一对应，避免“菜单能点击却找不到页面”。

## 六、根据当前路径反向高亮菜单

如果只用 `defaultSelectedKeys`，刷新、浏览器前进后退或手动输入地址时，高亮可能不更新。应读取实际路径：

```jsx
const location = useLocation()

<Menu
  items={items}
  selectedKeys={[location.pathname]}
  onClick={({ key }) => navigate(key)}
/>
```

这里 `selectedKeys` 是数组，`location.pathname` 是当前 URL 路径。菜单高亮由路由状态决定，保持了“地址是真实状态”的单向关系。

![菜单与路由对应](assets/day07/09.png)

## 七、获取用户信息并展示在 Header

用户资料会被多个页面使用，放在 Redux 的 `userInfo` 中：

```js
const userSlice = createSlice({
  name: 'user',
  initialState: { token: getToken() || '', userInfo: {} },
  reducers: {
    setUserInfo(state, action) {
      state.userInfo = action.payload
    },
  },
})

export const fetchUserInfo = () => async dispatch => {
  const res = await request.get('/user/profile')
  dispatch(setUserInfo(res.data))
}
```

```jsx
const dispatch = useDispatch()
const name = useSelector(state => state.user.userInfo.name)

useEffect(() => {
  dispatch(fetchUserInfo())
}, [dispatch])

<span className="user-name">{name || '用户'}</span>
```

请求头由拦截器自动携带 Token，页面只负责发起数据请求和展示结果。

## 八、退出登录与 Token 失效

用户主动退出时，先经 `Popconfirm` 确认，再清除 Redux 用户信息和本地 Token，最后返回登录页：

```jsx
const logout = () => {
  dispatch(clearUserInfo())
  navigate('/login', { replace: true })
}

<Popconfirm
  title="确认退出登录？"
  okText="退出"
  cancelText="取消"
  onConfirm={logout}
>
  <span>退出</span>
</Popconfirm>
```

被动失效发生在接口返回 401 时，可在响应拦截器集中处理：

```js
request.interceptors.response.use(
  response => response.data,
  error => {
    if (error.response?.status === 401) {
      removeToken()
      router.navigate('/login')
    }
    return Promise.reject(error)
  }
)
```

**区分两条链路**：`AuthRoute` 负责进入受保护页面前的检查；401 拦截器负责页面已进入后、Token 失效时的回退。访问令牌的有效性最终以服务端响应为准。

## 九、首页 ECharts：从基础图到可复用组件

课件先让图表跑通，再抽取可变参数：

```jsx
import { useEffect, useRef } from 'react'
import * as echarts from 'echarts'

function BarChart({ xData, sData, style = { width: 400, height: 300 } }) {
  const chartRef = useRef(null)

  useEffect(() => {
    const chart = echarts.init(chartRef.current)
    chart.setOption({
      xAxis: { type: 'category', data: xData },
      yAxis: { type: 'value' },
      series: [{ type: 'bar', data: sData }],
    })
    return () => chart.dispose()
  }, [xData, sData])

  return <div ref={chartRef} style={style} />
}
```

图表容器必须有宽高；`useEffect` 中初始化图表，卸载时 `dispose()` 清理实例。`xData`、`sData` 和尺寸作为 props 传入，就能复用同一个 BarChart。

![首页图表](assets/day07/13.png)

## 十、按业务模块封装 API

项目变大后，不应把所有 URL 和请求配置写在组件里。将统一 Axios 实例放在 `utils/request.js`，将文章和用户接口分别放在 `apis/article.js`、`apis/user.js`：

```js
// apis/user.js
export const loginAPI = form => request.post('/authorizations', form)
export const getProfileAPI = () => request.get('/user/profile')

// apis/article.js
export const getChannelAPI = () => request.get('/channels')
export const getArticleListAPI = params => request.get('/mp/articles', { params })
```

组件只调用语义化的 API 函数，接口路径变化时集中修改。`request` 负责通用 HTTP 配置，业务 API 负责具体端点，页面负责交互和渲染。

## 十一、排查与复习

- **接口一直 401**：查看 Network，请求头里是否有 `Authorization: Bearer <token>`；检查 Token 是否过期。
- **未登录仍看到 Layout**：确认路由元素被 `AuthRoute` 包裹，且本地旧 Token 是否还存在。
- **菜单亮错项**：`selectedKeys` 应从 `location.pathname` 计算，而不是固定默认值。
- **子页面空白**：检查 Layout 是否渲染 `<Outlet />`。
- **用户姓名为空**：检查资料请求、Redux reducer 注册名和 `useSelector` 路径。
- **图表不显示**：检查容器宽高、DOM ref、ECharts 初始化时机。
- **图表重复或内存泄漏**：Effect 清理函数调用 `dispose()`。

本日的关键是将登录后的用户身份、请求权限、页面路由和后台界面连接成一条可追踪的数据流。

## 十二、本日总结

第 07 天完成了后台的“壳”：Axios 自动带 Token，`AuthRoute` 阻止未登录访问，Layout 和 `Outlet` 承载子页面，Menu 根据路由导航并高亮，Redux 保存用户资料，退出和 401 处理清理身份状态，首页 ECharts 则展示基础数据。后续文章发布和列表都建立在这套壳上。